# Ⅰ第1回 演習2（E1-2）学習済みモデルの同時測定

3 つの学習済みモデル（A1：埋め込み平均＋MLP，A2：1次元畳み込み，A3：小型 Transformer）を動かし，
**正解率・損失・推論時間・メモリ** の 4 指標を同時に測る．
3 モデルはパラメータ数を約 30 万に揃えてあり，同じデータ（SST-2 映画レビューの肯定/否定）で同じ回数だけ学習してある．

実行時間の目安は 5 分以内．実行中は `waiting_task.md` の予測欄を埋める（結果を見る前に書く）．


## (0) 班と役割の設定

In [ ]:
# ===== (0) 班と役割の設定 =====
GROUP_ID = 1                  # ← 自分の班番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / auditor（5 人班は collector も）のいずれか．3 人班で recorder と auditor を兼任する人は "recorder"

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) コードテンプレート①：ライブラリのインポート

In [ ]:
import time
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from common import ASSETS, load_config, memory
from common.device import get_device, synchronize
from common.data import load_corpus, decode
from common.models import build_model, count_params
from common.logger import ResultLogger

## (2) コードテンプレート⑤：使用デバイスの選択

演習1 と同じ．`get_device()` はテンプレート⑤にメモリ上限を加えたもの．

In [ ]:
device = get_device()
print(f"使用デバイス: {device}")

## (3) コードテンプレート②③-テキスト：データ変換とデータセットの作成

教科書のテンプレート②（データ変換）と③（乱数固定とデータセット作成）に対応する．
画像では `transforms` が行う変換を，テキストでは「トークン化 → ID 化 → 固定長化」が担う（`common/data.py`．前週の `setup/download_assets.py` で実行済み）．
乱数固定の 3 行は教科書どおり毎回書く（今日は使わないが，第4回で再現性を検証する）．
評価には **テストデータ**（1,821 文．訓練にも検証にも使っていない文）を使う．

In [ ]:
# コードテンプレート③：乱数固定（教科書と同じ 3 行）
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

cfg = load_config()
corpus = load_corpus(cfg)
X_test = torch.from_numpy(corpus["X_test"])     # (1821, 40) トークン ID．0 はパディング
y_test = torch.from_numpy(corpus["y_test"])     # 0 = 否定, 1 = 肯定
print(f"訓練データ数: {len(corpus['X_train'])} 文")
print(f"検証データ数: {len(corpus['X_dev'])} 文")
print(f"テストデータ数: {len(X_test)} 文")
print("語彙:", corpus["vocab_size"], "| 系列長:", corpus["max_len"])
for i in range(3):
    print(y_test[i].item(), "|", decode(X_test[i], corpus["vocab"]))

## (4) コードテンプレート④⑥-計測用：ミニバッチと形状の確認

教科書のテンプレート④は `DataLoader` でミニバッチを作る．ここでは時間を測るときに **データ転送の時間を混ぜない** ため，先にテンソルをデバイスへ載せてから分割する（`DataLoader` を使うと毎回 CPU→GPU の転送が入り，モデルの速さではなく転送の速さを測ってしまう）．
テンプレート⑥にならい，ミニバッチの形状を確認する．

In [ ]:
# コードテンプレート④-計測用
BATCH_SIZE = 64
test_batches = [(X_test[i:i + BATCH_SIZE].to(device), y_test[i:i + BATCH_SIZE].to(device))
                for i in range(0, len(X_test), BATCH_SIZE)]

# コードテンプレート⑥：ミニバッチの形状確認
tokens, labels = test_batches[0]
print("ミニバッチサイズ:", len(tokens))
print("系列長:", tokens.shape[1])
print("ミニバッチ数:", len(test_batches))
print(tokens.shape)

## (5) コードテンプレート⑦-学習済みモデル：モデルのロードと評価モード

教科書のテンプレート⑦（モデル定義）の学習済み版．8 章「学習済みモデルのロード」と同じ手順で，重みファイルに入っている構造の情報（隠れ幅など）から同じ構造を組み立て，`load_state_dict` で学習済みパラメータを読み込む．
推論だけを行うので `model.eval()` で **評価モード** にする（Dropout などを無効化する）．

In [ ]:
def load_pretrained(arch):
    ckpt = torch.load(ASSETS / "weights" / f"day1_{arch}.pt", map_location="cpu")
    model, info = build_model(arch, ckpt["info"]["target"], ckpt["vocab_size"],
                              num_classes=ckpt["num_classes"], max_len=ckpt["max_len"], d=ckpt["embed_dim"])
    # TODO: ckpt["state_dict"] を load_state_dict でモデルにロードし，評価モードにする
    ...
    return model.to(device), info

for arch in ["A1", "A2", "A3"]:
    m, info = load_pretrained(arch)
    print(arch, "params =", f"{count_params(m):,}", "| hidden =", info["hidden"])

## (6) コードテンプレート⑧⑫-4指標：損失関数と正解率

教科書のテンプレート⑧（損失関数 `criterion`）と⑫（テストデータでの評価）をそのまま使い，正解率（Accuracy）に加えて損失も集計する．
正解率は「正解した割合」，損失は「正解ラベルに対する確信の低さ」（交差エントロピー）．**同じモデルでも 2 つは違う順位になりうる**．

In [ ]:
criterion = nn.CrossEntropyLoss(reduction="sum")

def evaluate(model, batches):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    with torch.no_grad():
        for tokens, labels in batches:
            outputs = model(tokens)                       # (B, 2)
            total_loss += criterion(outputs, labels).item()
            # TODO: 教科書のテンプレート⑫と同じ 3 行を書く（torch.max(outputs.data, 1) の 2 つ目の戻り値が予測クラス）
            ...
    assert total > 0, "TODO 未実装: ループ内で total と correct を更新すること"
    return correct / total, total_loss / total

model, _ = load_pretrained("A1")
acc, loss = evaluate(model, test_batches)
print(f"A1: Accuracy: {acc*100:.2f}%  Loss: {loss:.4f}")

## (7) コードテンプレート⑬-計測：推論時間

演習1 で見たとおり **同期** が必要である．また 1 回では変動する（GPU のクロックが上下する）ので，
**最低 2 秒ぶん繰り返して中央値** を取る（演習1 の `timed_loop` と同じ作法）．
「1000 文あたりの ms」（スループット）と「1 文だけ入れたときの ms」（レイテンシ）は **別の指標** である．

In [ ]:
def measure_time(model, batches, min_sec=2.0, min_repeats=5):
    """全テストデータを流す時間 → 1000 文あたりの ms（中央値）"""
    model.eval()
    n = sum(len(t) for t, _ in batches)
    times = []
    with torch.no_grad():
        for tokens, _ in batches:                     # ウォームアップ 1 周
            model(tokens)
        synchronize(device)
        t_start = time.perf_counter()
        while len(times) < min_repeats or time.perf_counter() - t_start < min_sec:
            t0 = time.perf_counter()
            for tokens, _ in batches:
                model(tokens)
            # TODO: 時計を読む前に同期する
            times.append(time.perf_counter() - t0)
    return float(np.median(times)) / n * 1000 * 1000

def measure_latency(model, batches, n=200):
    """1 文ずつ入れたときの 1 文あたり ms（中央値）"""
    model.eval()
    X = batches[0][0]                                 # 先頭バッチから 1 文ずつ取り出す（デバイス上）
    times = []
    with torch.no_grad():
        for i in range(n):
            x = X[i % len(X):i % len(X) + 1]
            synchronize(device)
            t0 = time.perf_counter()
            model(x)
            # TODO: 時計を読む前に同期する
            times.append(time.perf_counter() - t0)
    return float(np.median(times)) * 1000

print(f"A1: {measure_time(model, test_batches):.2f} ms / 1000文,  {measure_latency(model, test_batches):.3f} ms / 1文")

## (8) コードテンプレート⑬-計測：メモリ

「モデルの大きさ」（重みのバイト数）と「推論中のピーク」（途中の計算結果が占める量）は別物．
3 モデルはパラメータ数が同じなので **重みの大きさは同じ**．違いが出るのは推論中のピークである．
`forward_peak_mb` は各層の出力直後に使用量を読み，最大値を返す（裏でサンプリングする方式は数十 ms の処理では取りこぼす）．

In [ ]:
def measure_memory(model, batches):
    model_mb = count_params(model) * 4 / 2**20              # float32 = 4 バイト
    def run():
        with torch.no_grad():
            for tokens, _ in batches:
                model(tokens)
    peak_mb = memory.forward_peak_mb(model, run)             # 推論中に増えた分のピーク
    return model_mb, peak_mb

print("A1: model %.2f MB, peak %.2f MB" % measure_memory(model, test_batches))

## (9) コードテンプレート⑫-4指標：3 モデルの同時測定

同じ関数で 3 モデルを測り，1 枚の表にする．
GPU のブースト（演習1）で最初に測るモデルだけ有利にならないよう，測定前に 2 秒 GPU を回して定常状態にしてから始める．

In [ ]:
from common.bench import warm_up
warm_up(device, sec=2.0)                       # ブースト状態を抜けさせる

rows = []
for arch in ["A1", "A2", "A3"]:
    model, info = load_pretrained(arch)
    t0 = time.perf_counter()
    acc, loss = evaluate(model, test_batches)
    ms1000 = measure_time(model, test_batches)
    lat = measure_latency(model, test_batches)
    model_mb, peak_mb = measure_memory(model, test_batches)
    rows.append({"model": arch, "params": count_params(model),
                 "accuracy": acc, "loss": loss,
                 "infer_ms_per_1000": ms1000, "latency_ms_b1": lat,
                 "model_mb": model_mb, "peak_mem_mb": peak_mb,
                 "elapsed_sec": time.perf_counter() - t0})
table = pd.DataFrame(rows).set_index("model")
table.round(4)

## (10) 予測との照合

`waiting_task.md` に書いた予測（最も正解率が高いのは？最も速いのは？）と表を比較する．
**外れた予測** をプレゼンで話す．当たった予測より報告の価値がある．

In [ ]:
print("最も正解率が高い:", table["accuracy"].idxmax())
print("最も損失が低い:", table["loss"].idxmin())
print("最も速い(1000文):", table["infer_ms_per_1000"].idxmin())
print("最も速い(1文)  :", table["latency_ms_b1"].idxmin())
print("最も省メモリ    :", table["peak_mem_mb"].idxmin())

## (11) コードテンプレート⑬-記録：CSV に記録

演習3 と第7回（改善のベースライン）がこの CSV を読む．

In [ ]:
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d1", exercise="ex2", device=device)
for arch, r in table.iterrows():
    logger.log_many({k: r[k] for k in ["accuracy", "loss", "infer_ms_per_1000", "latency_ms_b1",
                                       "model_mb", "peak_mem_mb", "params"]},
                    condition=arch, seed=0, elapsed_sec=r["elapsed_sec"])
table.to_csv(logger.path.with_name(f"table_c1_d1_ex2_{logger.group_id}.csv"))
print("書き込み先:", logger.path)
pd.read_csv(logger.path).tail(7)

## (12) グループディスカッション（5 分）→ グループ内プレゼン2（1 人 2 分）

`waiting_task.md` の予測表を全員分並べ，**外れた予測から** 話す．全員が 1 回は発言する．

**討議の問い**
1. 正解率 1 位と損失 1 位は同じモデルか．違うなら，損失は何を余分に見ているか
2. 1000 文あたりの時間と 1 文の時間で速いモデルは同じか．違うなら，どちらの指標が「実運用」に近いか
3. パラメータ数が同じなのに推論中のピークメモリが違うのはなぜか（各モデルの forward を見て答える）
4. 正解率の差 0.4 ポイントは「差がある」と言えるか．言えないなら，何を測れば言えるようになるか（→ 第4回）

**プレゼン2**：verifier は条件を変えた再実験（バッチサイズ，繰り返し回数）で何が変わったかを画面で見せる．

In [ ]:
print("=== 討議用: 4 指標の順位（1 = 最良）===")
ranks = pd.DataFrame({
    "accuracy":          table["accuracy"].rank(ascending=False).astype(int),
    "loss":              table["loss"].rank().astype(int),
    "infer_ms_per_1000": table["infer_ms_per_1000"].rank().astype(int),
    "latency_ms_b1":     table["latency_ms_b1"].rank().astype(int),
    "peak_mem_mb":       table["peak_mem_mb"].rank().astype(int),
})
display(ranks)
print("指標ごとに 1 位が違う数:", ranks.idxmin().nunique(), "モデル")
print("正解率の最大差:", f"{(table['accuracy'].max() - table['accuracy'].min())*100:.2f} ポイント（3 シードのばらつきは約 0.2〜0.7 ポイント）")
print("討議メモ（recorder）: 外れた予測 ________ / 理由の仮説 ________ / 次に測るべきもの ________")